In [1]:
import sys
from pathlib import Path

parent_dir = str(Path().resolve().parents[1])
sys.path.insert(0, parent_dir)

In [2]:
from src.paths import DATA_DIR
from src.Qlassifier.baseline import run_tf_idf

In [3]:
subjects = ["chemistry", "specialist_mathematics"]
exams = ["2023", "2023_2"]

dfs = []
for subject, exam in zip(subjects, exams):
    subject_path = DATA_DIR / subject
    exam_path = subject_path / "past_exams" / f"{exam}.pdf"
    df = run_tf_idf(exam_path)
    dfs.append(df)

## Evaluation

Since I am no expert in chemistry, I supplied GPT-5 with the necessary documents and asked it to hand-label the best topic classificaitons for us. Unfortunately, for whatever reason, gpt-5 could only reliably label MCQ questions and wasn't very helpful for short answer ones. But alas, even the mcq was of very low quality - by my calculations even worse than this rudimentary TF-IDF approach - as i detected with my limited chemistry knowledge. So, I will hand label on this occasion to the best of my ability, with the help of GPT-5 on individual questions, one at a time.

In [ ]:
true_topic_indices = [ # labelled -1 if i dont know
    0, 2, 6, 5, 10, 9, 0, 4, 0, 4,
    3, 3, 6, 4, -1, 9, 1, 9, 6, 8,
    10 ,2, 6, 2, 5, 11, 12, 2, 9, 2,
]

chem_df = dfs[0]
mcq_chem = chem_df.loc[:29, :].copy()
mcq_chem["True Topic"] = true_topic_indices

mcq_chem.head(5)

In [7]:
accuracy = sum(mcq_chem["Predicted Topic"] == mcq_chem["True Topic"]) / mcq_chem.shape[0]
print(f"Accuracy on chemistry MCQ {100*accuracy:.1f}%")

Accuracy on chemistry MCQ 73.3%


But our goal isn't really to JUST assign a question to a topic; this is a flawed problem design, since questions may belong to multiple topics. The hand labelling above was just what I deemed to be the single BEST answer. In reality, its the distribution of the weights for each topic that we are interested in. We also didn't test short answer (since the labelling is tedious to say the least, and the one "true label" is even more ambiguous). But this accuracy metric serves as a decent baseline regardless. 

We move on to a mathematics subject, in which we expect our 'model' to perform FAR worse, since we don't have LaTeX data and plenty of the context is in the LaTeX for math exams, especially relative to chemistry ones. This one I can hand label reliably.

In [ ]:
ground_truth = [
    0, 1, 1, 2, 2, 4, 4, 4, 7, 3, 
    3, 5, 5, 6, 6, 8, 6, 7, 9, 11
]

math_df = dfs[1]
mcq_math = math_df.loc[:19, :].copy()
mcq_math["True Topic"] = ground_truth
mcq_math.head(5)

In [19]:
accuracy = sum(mcq_math["Predicted Topic"] == mcq_math["True Topic"]) / mcq_math.shape[0]
print(f"Accuracy on specialist math MCQ {100*accuracy:.1f}%")

Accuracy on specialist math MCQ 45.0%


As expected! These resutls will serve as our baseline. 